<!--nav--> [🗺 Learning path](README.md) · **14/20** · ◀ [GRPO Reasoning Training](./GRPO_Reasoning_Training.ipynb) · [Multi Trace Agent Evaluation](./Multi_Trace_Agent_Evaluation.ipynb) ▶

# LLM-as-Judge Evaluation (Self-Contained)

## What is LLM-as-Judge?

Instead of humans rating model outputs, we use an **LLM** to judge quality.

```
Test Prompt --> Model (base) -----> Response A ----\
                                                    --> Same Model as Judge --> Winner
Test Prompt --> Model (trained) --> Response B ----/
```

### This notebook uses NO external APIs
We use the **same models** (TinyLlama, LLaVA) as both the generator AND the judge.

| Part | Model | What we compare |
|------|-------|----------------|
| **Part 1** | TinyLlama 1.1B (text) | Base vs DPO-aligned |
| **Part 2** | LLaVA 7B (multimodal) | Base vs QLoRA-trained |

### Three evaluation methods (all local, no API)
1. **Automated metrics** — word count, vocabulary diversity, specificity
2. **Rule-based scoring** — rubric checks for detail, structure, accuracy
3. **Model-as-Judge** — prompt the model itself to compare and score responses

> **Note:** Using a small model as its own judge has biases (self-preference,
> limited reasoning). In production, you'd use a stronger model (GPT-4, Claude, Gemini).
> This notebook demonstrates the *technique* — swap in a stronger judge for better results.

---
**Runtime:** T4 GPU (Runtime > Change runtime type > T4 GPU)

## Step 1: Install & Setup

In [ ]:
!pip install -q transformers trl datasets peft accelerate matplotlib
!pip install -q -U "bitsandbytes>=0.46.1"

# Verify bitsandbytes installed correctly
import importlib
if importlib.util.find_spec("bitsandbytes") is None:
    print("\n*** bitsandbytes not found! Restart runtime: Runtime > Restart session ***")
    print("*** Then re-run all cells from the top. ***")
else:
    import bitsandbytes
    print("bitsandbytes version:", bitsandbytes.__version__)

In [ ]:
import torch
import gc
import time
import re
import numpy as np
import os
os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4"
print("GPU:", torch.cuda.get_device_name(0))
print("Memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

eval_results = {"text": {}, "multimodal": {}}

---
# PART 1: Text Model Evaluation

## TinyLlama 1.1B — Base vs DPO-Aligned

1. Generate responses from **base** model
2. Quick DPO train, generate **aligned** responses
3. Use the **DPO model itself** as a judge to compare

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TEST_PROMPTS = [
    "Explain what a black hole is.",
    "How do I make scrambled eggs?",
    "What is machine learning?",
    "Why is exercise important?",
    "Explain recursion in programming.",
    "What is gravity?",
    "How do I learn Python?",
    "What makes a good friend?",
    "Why do we dream?",
    "How does the internet work?",
]

def generate_response(model, prompt, max_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_tokens,
            temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("%d test prompts ready." % len(TEST_PROMPTS))

### Generate BASE model responses

In [ ]:
print("Loading base model...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)

base_responses = []
print("Generating base model responses...\n")
for i, prompt in enumerate(TEST_PROMPTS):
    resp = generate_response(model_base, prompt)
    base_responses.append(resp)
    print("[%d/%d] %s" % (i + 1, len(TEST_PROMPTS), prompt))
    print("  >> %s\n" % resp[:150])

del model_base
clear_gpu()
print("Base responses collected. GPU cleared.")

### DPO Training + Generate Aligned Responses

In [ ]:
from datasets import Dataset
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

PREFERENCE_DATA = [
    {
        "prompt": "Explain what a black hole is.",
        "chosen": "A black hole is a region in space where gravity is so "
                  "strong that nothing, not even light, can escape. They form "
                  "when massive stars collapse at the end of their life cycle.",
        "rejected": "A black hole is a hole that is black. It sucks things in. "
                     "Nobody really knows what they are.",
    },
    {
        "prompt": "How do I make scrambled eggs?",
        "chosen": "Crack 2-3 eggs into a bowl, whisk with salt and pepper. "
                  "Heat butter in a pan over medium-low heat, pour in eggs, "
                  "and gently stir with a spatula until softly set.",
        "rejected": "Put eggs in pan. Cook them. Add stuff if you want.",
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of AI where computers learn "
                  "patterns from data instead of being explicitly programmed. "
                  "For example, a spam filter learns from labeled emails.",
        "rejected": "Machine learning is when computers learn stuff. "
                     "It's really complicated and uses lots of math.",
    },
    {
        "prompt": "Why is exercise important?",
        "chosen": "Regular exercise strengthens your heart, improves mood by "
                  "releasing endorphins, helps maintain a healthy weight, and "
                  "reduces the risk of chronic diseases like diabetes.",
        "rejected": "Exercise is good for you. You should do it because "
                     "everyone says so.",
    },
    {
        "prompt": "Explain recursion in programming.",
        "chosen": "Recursion is when a function calls itself to solve smaller "
                  "sub-problems. For example, factorial(5) = 5 * factorial(4). "
                  "Every recursive function needs a base case to stop.",
        "rejected": "Recursion is a hard concept. It's when things repeat. "
                     "You'll understand it eventually.",
    },
]

def format_as_chat(prompt, response):
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]

rows = []
for ex in PREFERENCE_DATA:
    rows.append({
        "prompt": [{"role": "user", "content": ex["prompt"]}],
        "chosen": format_as_chat(ex["prompt"], ex["chosen"]),
        "rejected": format_as_chat(ex["prompt"], ex["rejected"]),
    })
dataset = Dataset.from_list(rows)
print("DPO dataset: %d preference pairs" % len(dataset))

In [ ]:
print("Loading model for DPO training...")
model_dpo = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir="./eval_dpo_output",
    beta=0.1, num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5, logging_steps=1,
    bf16=True, gradient_checkpointing=True,
    do_eval=False, remove_unused_columns=False,
    report_to="none", save_strategy="no",
)

trainer = DPOTrainer(
    model=model_dpo, args=training_args,
    train_dataset=dataset, processing_class=tokenizer,
    peft_config=lora_config,
)

print("Training DPO...")
trainer.train()
print("DPO training complete!")
del trainer

# Generate aligned responses
model_dpo.eval()
dpo_responses = []
print("\nGenerating DPO model responses...\n")
for i, prompt in enumerate(TEST_PROMPTS):
    resp = generate_response(model_dpo, prompt)
    dpo_responses.append(resp)
    print("[%d/%d] %s" % (i + 1, len(TEST_PROMPTS), prompt))
    print("  >> %s\n" % resp[:150])

---
## Evaluation Method 1: Automated Metrics

Computed directly from text — no model needed.

| Metric | What it measures |
|--------|------------------|
| **Word Count** | Response detail/length |
| **Unique Words %** | Vocabulary diversity |
| **Sentence Count** | Structure |
| **Specificity** | Concrete details (numbers, examples, technical terms) |

In [ ]:
def compute_text_metrics(response):
    words = response.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', response) if s.strip()]
    unique_words = set(w.lower() for w in words)

    specificity = 0
    specificity += len(re.findall(r'\d+', response))  # numbers
    specificity += len(re.findall(r'for example|such as|e\.g\.|for instance|like ', response, re.I))  # examples
    specificity += sum(1 for w in words if len(w) > 8)  # technical terms
    specificity += len(re.findall(r'because|therefore|this means|which causes', response, re.I))  # causal

    return {
        "word_count": len(words),
        "sentence_count": len(sentences),
        "unique_word_pct": 100.0 * len(unique_words) / max(len(words), 1),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "specificity": specificity,
    }


base_metrics = [compute_text_metrics(r) for r in base_responses]
dpo_metrics = [compute_text_metrics(r) for r in dpo_responses]

metric_names = ["word_count", "sentence_count", "unique_word_pct", "avg_word_length", "specificity"]
print("%-20s %12s %12s %10s" % ("Metric", "Base", "DPO", "Change"))
print("-" * 58)

avg_base = {}
avg_dpo = {}
for name in metric_names:
    b = np.mean([m[name] for m in base_metrics])
    d = np.mean([m[name] for m in dpo_metrics])
    avg_base[name] = b
    avg_dpo[name] = d
    change = ((d - b) / max(abs(b), 0.01)) * 100
    arrow = "+" if change > 0 else ""
    print("%-20s %12.1f %12.1f %9s%.0f%%" % (name, b, d, arrow, change))

eval_results["text"]["base_metrics"] = avg_base
eval_results["text"]["dpo_metrics"] = avg_dpo

## Evaluation Method 2: Rule-Based Scoring

A rubric that scores responses 0-10 on 5 dimensions.

In [ ]:
def rule_based_score(response, prompt):
    words = response.split()
    lower = response.lower()
    sentences = [s.strip() for s in re.split(r'[.!?]+', response) if s.strip()]
    scores = {}

    # COMPLETENESS
    if len(words) >= 80: scores["completeness"] = 10
    elif len(words) >= 50: scores["completeness"] = 8
    elif len(words) >= 30: scores["completeness"] = 6
    elif len(words) >= 15: scores["completeness"] = 4
    else: scores["completeness"] = 2

    # SPECIFICITY
    s = 3
    s += min(3, len(re.findall(r'\d+', response)))
    s += min(2, len(re.findall(r'for example|such as|like |e\.g\.', lower)))
    s += min(2, sum(1 for w in words if len(w) > 8) // 2)
    scores["specificity"] = min(10, s)

    # STRUCTURE
    s = 3
    if len(sentences) >= 3: s += 2
    if len(sentences) >= 5: s += 1
    s += min(3, len(re.findall(r'first|second|also|however|moreover|additionally|finally', lower)))
    if re.search(r'\d\.|\d\)|\-\s', response): s += 1
    scores["structure"] = min(10, s)

    # ACCURACY SIGNALS
    s = 5
    s -= min(3, len(re.findall(r'maybe|probably|i think|sort of|kind of|basically|really|stuff|things', lower)))
    s += min(4, len(re.findall(r'because|therefore|which means|this is|defined as|refers to', lower)))
    if re.search(r'nobody knows|too complicated|you.ll understand|just google', lower): s -= 3
    scores["accuracy_signals"] = max(0, min(10, s))

    # HELPFULNESS
    s = 4
    prompt_words = set(prompt.lower().split()) - {"what", "is", "how", "do", "i", "a", "the", "in", "why"}
    s += min(3, sum(1 for w in prompt_words if w in lower))
    s += min(2, len(re.findall(r'you can|try |start |learn |use |make |create ', lower)))
    if len(words) < 10: s -= 2
    scores["helpfulness"] = max(0, min(10, s))

    scores["total"] = sum(v for k, v in scores.items()) / 5.0
    return scores


base_scores = [rule_based_score(r, p) for r, p in zip(base_responses, TEST_PROMPTS)]
dpo_scores = [rule_based_score(r, p) for r, p in zip(dpo_responses, TEST_PROMPTS)]

dimensions = ["completeness", "specificity", "structure", "accuracy_signals", "helpfulness", "total"]
print("%-20s %8s %8s %8s" % ("Dimension", "Base", "DPO", "Winner"))
print("-" * 48)

base_avgs = {}
dpo_avgs = {}
for dim in dimensions:
    b = np.mean([s[dim] for s in base_scores])
    d = np.mean([s[dim] for s in dpo_scores])
    base_avgs[dim] = b
    dpo_avgs[dim] = d
    winner = "DPO" if d > b else ("Base" if b > d else "Tie")
    print("%-20s %8.1f %8.1f %8s" % (dim, b, d, winner))

eval_results["text"]["base_rubric"] = base_avgs
eval_results["text"]["dpo_rubric"] = dpo_avgs

In [ ]:
# Radar chart
import matplotlib.pyplot as plt

plt.style.use('dark_background')

dims = ["completeness", "specificity", "structure", "accuracy_signals", "helpfulness"]
labels = ["Complete", "Specific", "Structured", "Accurate", "Helpful"]

base_vals = [base_avgs[d] for d in dims]
dpo_vals = [dpo_avgs[d] for d in dims]

angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
base_plot = base_vals + [base_vals[0]]
dpo_plot = dpo_vals + [dpo_vals[0]]
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.fill(angles, base_plot, color="#f87171", alpha=0.15)
ax.plot(angles, base_plot, color="#f87171", linewidth=2, marker="o", label="Base")
ax.fill(angles, dpo_plot, color="#34d399", alpha=0.15)
ax.plot(angles, dpo_plot, color="#34d399", linewidth=2, marker="o", label="DPO")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 10)
ax.set_title("Rule-Based Evaluation: Base vs DPO", fontsize=16, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=12)
plt.tight_layout()
plt.show()

## Evaluation Method 3: Model-as-Judge

We use the **DPO-trained TinyLlama itself** as a judge.

The model sees both responses (anonymized as A and B, randomly ordered)
and picks which is better.

### How it works
```
Judge Prompt:
  "Here are two responses to the question '...'
   Response A: ...
   Response B: ...
   Which is better? Answer with just A or B."
```

### Limitations of self-judging
- Small models are worse judges than large ones
- Self-preference bias (model prefers its own style)
- We mitigate position bias by **randomly swapping A/B order**

In [ ]:
import random

JUDGE_TEMPLATE = """You are a helpful judge. Compare two responses to a question and decide which is better.

Question: {prompt}

Response A:
{response_a}

Response B:
{response_b}

Which response is better? Consider accuracy, detail, helpfulness, and clarity.
Answer with ONLY the letter A or B."""


def judge_with_model(model, prompt, resp_base, resp_dpo):
    """Use the model itself to judge two responses.
    Returns 'Base', 'DPO', or 'Tie' after random swap to avoid position bias."""
    swap = random.random() < 0.5
    if swap:
        response_a, response_b = resp_dpo, resp_base
    else:
        response_a, response_b = resp_base, resp_dpo

    judge_prompt = JUDGE_TEMPLATE.format(
        prompt=prompt,
        response_a=response_a[:300],
        response_b=response_b[:300],
    )

    messages = [{"role": "user", "content": judge_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()

    # Parse: look for A or B in the answer
    if "A" in answer and "B" not in answer:
        raw = "A"
    elif "B" in answer and "A" not in answer:
        raw = "B"
    else:
        return "Tie", answer

    # Unswap
    if swap:
        winner = "DPO" if raw == "A" else "Base"
    else:
        winner = "Base" if raw == "A" else "DPO"

    return winner, answer


print("Judge function ready.")
print("Position bias mitigation: A/B order is randomly swapped per prompt.")

In [ ]:
# Run the model as judge on all test prompts
random.seed(42)
judge_results = []
wins = {"Base": 0, "DPO": 0, "Tie": 0}

print("Model judging %d pairs...\n" % len(TEST_PROMPTS))

for i, prompt in enumerate(TEST_PROMPTS):
    winner, raw = judge_with_model(model_dpo, prompt, base_responses[i], dpo_responses[i])
    judge_results.append({"prompt": prompt, "winner": winner, "raw": raw})
    wins[winner] = wins.get(winner, 0) + 1
    print("[%d/%d] %s" % (i + 1, len(TEST_PROMPTS), prompt))
    print("  Winner: %s (raw: %s)" % (winner, raw))

total = len(judge_results)
print("\n" + "=" * 50)
print("MODEL-AS-JUDGE RESULTS (TinyLlama DPO)")
print("=" * 50)
print("Base wins: %d/%d (%.0f%%)" % (wins["Base"], total, 100 * wins["Base"] / total))
print("DPO wins:  %d/%d (%.0f%%)" % (wins["DPO"], total, 100 * wins["DPO"] / total))
print("Ties:      %d/%d (%.0f%%)" % (wins["Tie"], total, 100 * wins["Tie"] / total))

eval_results["text"]["judge_wins"] = wins

In [ ]:
# Perplexity comparison: how "confident" is the model about each response?
# Lower perplexity = model finds the text more natural/likely

def compute_perplexity(model, text):
    """Compute perplexity of text under the model."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

base_ppls = []
dpo_ppls = []

print("Computing perplexity (model confidence in each response)...\n")
for i, prompt in enumerate(TEST_PROMPTS):
    # Format as full conversation for perplexity
    base_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": base_responses[i]}],
        tokenize=False,
    )
    dpo_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": dpo_responses[i]}],
        tokenize=False,
    )
    bp = compute_perplexity(model_dpo, base_text)
    dp = compute_perplexity(model_dpo, dpo_text)
    base_ppls.append(bp)
    dpo_ppls.append(dp)
    print("[%d] %s" % (i + 1, prompt[:50]))
    print("  Base PPL: %.1f | DPO PPL: %.1f | Prefers: %s" % (bp, dp, "DPO" if dp < bp else "Base"))

print("\nAvg Perplexity — Base: %.1f | DPO: %.1f" % (np.mean(base_ppls), np.mean(dpo_ppls)))
print("Lower = model finds text more natural. DPO model should prefer DPO responses.")

eval_results["text"]["base_ppl"] = float(np.mean(base_ppls))
eval_results["text"]["dpo_ppl"] = float(np.mean(dpo_ppls))

In [ ]:
# Combined text evaluation visualization
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Win rate pie
win_labels = []
win_vals = []
win_colors = []
color_map = {"Base": "#f87171", "DPO": "#34d399", "Tie": "#818cf8"}
for k in ["Base", "DPO", "Tie"]:
    if wins.get(k, 0) > 0:
        win_labels.append("%s (%d)" % (k, wins[k]))
        win_vals.append(wins[k])
        win_colors.append(color_map[k])

axes[0].pie(win_vals, labels=win_labels, colors=win_colors,
            autopct="%.0f%%", textprops={"fontsize": 13, "color": "white"},
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1})
axes[0].set_title("Model-as-Judge Win Rate", fontsize=14)

# 2. Perplexity comparison
x = np.arange(len(TEST_PROMPTS))
axes[1].bar(x - 0.2, base_ppls, 0.4, color="#f87171", label="Base resp", alpha=0.8)
axes[1].bar(x + 0.2, dpo_ppls, 0.4, color="#34d399", label="DPO resp", alpha=0.8)
axes[1].set_title("Perplexity (lower = more natural)", fontsize=14)
axes[1].set_xlabel("Test Prompt #")
axes[1].set_ylabel("Perplexity")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.2, axis="y")

# 3. Automated metrics comparison
metric_labels = ["Words", "Sentences", "Unique%", "Specificity"]
metric_keys = ["word_count", "sentence_count", "unique_word_pct", "specificity"]
b_vals = [avg_base[k] for k in metric_keys]
d_vals = [avg_dpo[k] for k in metric_keys]

x2 = np.arange(len(metric_labels))
axes[2].bar(x2 - 0.2, b_vals, 0.4, color="#f87171", label="Base", alpha=0.8)
axes[2].bar(x2 + 0.2, d_vals, 0.4, color="#34d399", label="DPO", alpha=0.8)
axes[2].set_xticks(x2)
axes[2].set_xticklabels(metric_labels, fontsize=11)
axes[2].set_title("Automated Metrics", fontsize=14)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.show()

## Side-by-Side Response Comparison

In [ ]:
from IPython.display import HTML, display

cards = ""
for i, prompt in enumerate(TEST_PROMPTS):
    winner = judge_results[i]["winner"]
    base_r = base_responses[i].replace("<", "&lt;").replace(">", "&gt;")
    dpo_r = dpo_responses[i].replace("<", "&lt;").replace(">", "&gt;")
    b_score = rule_based_score(base_responses[i], prompt)["total"]
    d_score = rule_based_score(dpo_responses[i], prompt)["total"]
    winner_color = "#34d399" if winner == "DPO" else ("#f87171" if winner == "Base" else "#818cf8")

    cards += (
        '<div style="margin:12px 0;padding:15px;background:#1a1a2e;border-radius:10px;border:1px solid #333;">'
        '<div style="color:#a78bfa;font-weight:bold;font-size:14px;margin-bottom:10px;">'
        + str(i + 1) + '. ' + prompt + '</div>'
        '<div style="display:flex;gap:15px;">'
        '<div style="flex:1;padding:10px;background:#0d1117;border-radius:8px;border-left:3px solid #f87171;">'
        '<div style="color:#f87171;font-weight:bold;margin-bottom:5px;">Base (score: ' + ("%.1f" % b_score) + '/10)</div>'
        '<div style="color:#ccc;font-size:12px;">' + base_r[:400] + '</div></div>'
        '<div style="flex:1;padding:10px;background:#0d1117;border-radius:8px;border-left:3px solid #34d399;">'
        '<div style="color:#34d399;font-weight:bold;margin-bottom:5px;">DPO (score: ' + ("%.1f" % d_score) + '/10)</div>'
        '<div style="color:#ccc;font-size:12px;">' + dpo_r[:400] + '</div></div>'
        '</div>'
        '<div style="margin-top:8px;text-align:center;">'
        '<span style="color:' + winner_color + ';font-weight:bold;font-size:13px;">'
        'Model Judge: ' + winner + ' | '
        'PPL: Base=' + ("%.0f" % base_ppls[i]) + ' DPO=' + ("%.0f" % dpo_ppls[i])
        + '</span></div></div>'
    )

display(HTML(
    '<div style="font-family:sans-serif;">'
    '<h2 style="color:#a78bfa;">Text: Base vs DPO — All 10 Prompts</h2>'
    + cards + '</div>'
))

In [ ]:
# Free text model before multimodal
del model_dpo
clear_gpu()
print("Text model freed. Ready for multimodal evaluation.")

---
# PART 2: Multimodal Model Evaluation

## LLaVA 7B — Base vs QLoRA-Trained

1. Generate image descriptions from **base** LLaVA
2. Quick QLoRA train on anime captions, generate **trained** descriptions
3. Compare with image-specific metrics + model-as-judge

In [ ]:
from datasets import load_dataset
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig as LoraConfig2, get_peft_model, prepare_model_for_kbit_training

clear_gpu()

MM_MODEL = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(MM_MODEL)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

raw_data = load_dataset("lambdalabs/naruto-blip-captions", split="train")
raw_data = raw_data.shuffle(seed=42)
test_images = raw_data.select(range(250, 260))  # 10 test images

print("Test images: %d" % len(test_images))

In [ ]:
def generate_mm_response(model, image, prompt_text="Describe this image in detail."):
    prompt = "USER: <image>\n" + prompt_text + "\nASSISTANT:"
    inputs = processor(text=prompt, images=image.convert("RGB"), return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=150,
            temperature=0.7, do_sample=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    return processor.tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

print("Generation function ready.")

### Generate BASE LLaVA descriptions

In [ ]:
print("Loading base LLaVA (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_base_mm = LlavaForConditionalGeneration.from_pretrained(
    MM_MODEL, quantization_config=bnb_config, device_map="auto",
)

base_mm_responses = []
real_captions = []
print("Generating base descriptions...\n")
for i in range(len(test_images)):
    img = test_images[i]["image"]
    caption = test_images[i]["text"]
    real_captions.append(caption)
    resp = generate_mm_response(model_base_mm, img)
    base_mm_responses.append(resp)
    print("[%d/%d] Real: %s" % (i + 1, len(test_images), caption[:80]))
    print("        Base: %s\n" % resp[:100])

del model_base_mm
clear_gpu()
print("Base descriptions collected. GPU cleared.")

### QLoRA Training + Generate Trained Descriptions

In [ ]:
from transformers import TrainingArguments, Trainer

print("Loading model for QLoRA training...")

model_qlora_mm = LlavaForConditionalGeneration.from_pretrained(
    MM_MODEL, quantization_config=bnb_config, device_map="auto",
)
model_qlora_mm = prepare_model_for_kbit_training(model_qlora_mm)

qlora_config = LoraConfig2(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model_qlora_mm = get_peft_model(model_qlora_mm, qlora_config)
model_qlora_mm.print_trainable_parameters()

In [ ]:
PROMPT_TEMPLATE = "USER: <image>\nDescribe this image in detail.\nASSISTANT: %s</s>"
sft_data = raw_data.select(range(200))

def format_for_llava(example):
    example["formatted_text"] = PROMPT_TEMPLATE % example["text"]
    return example

sft_dataset = sft_data.map(format_for_llava)

class LLaVACollator:
    def __init__(self, proc):
        self.proc = proc
    def __call__(self, examples):
        texts = [ex["formatted_text"] for ex in examples]
        images = [ex["image"].convert("RGB") for ex in examples]
        batch = self.proc(text=texts, images=images, return_tensors="pt", padding=True)
        if "pixel_values" in batch:
            batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)
        labels = batch["input_ids"].clone()
        labels[labels == self.proc.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
        return batch

collator = LLaVACollator(processor)

training_args_mm = TrainingArguments(
    output_dir="./eval_qlora_mm",
    max_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer_mm = Trainer(
    model=model_qlora_mm,
    args=training_args_mm,
    train_dataset=sft_dataset,
    data_collator=collator,
)

print("Training QLoRA...")
trainer_mm.train()
print("Training complete!")
del trainer_mm

In [ ]:
model_qlora_mm.eval()
trained_mm_responses = []

print("Generating trained descriptions...\n")
for i in range(len(test_images)):
    img = test_images[i]["image"]
    resp = generate_mm_response(model_qlora_mm, img)
    trained_mm_responses.append(resp)
    print("[%d/%d] Real:    %s" % (i + 1, len(test_images), real_captions[i][:80]))
    print("        Trained: %s\n" % resp[:100])

### Multimodal Automated Metrics

In [ ]:
def compute_caption_metrics(generated, reference):
    gen_words = set(generated.lower().split())
    ref_words = set(reference.lower().split())
    stopwords = {"a", "an", "the", "is", "of", "in", "and", "to", "with", "on", "that", "it", "for", "as", "are", "was", "this"}
    gen_content = gen_words - stopwords
    ref_content = ref_words - stopwords

    overlap = len(gen_content & ref_content) / max(len(ref_content), 1)

    color_words = {"red", "blue", "green", "yellow", "black", "white", "orange",
                   "purple", "pink", "brown", "gray", "grey", "blonde", "golden"}
    colors = len(gen_words & color_words)

    details = len(re.findall(
        r'wearing|holding|standing|sitting|background|foreground|'
        r'hair|eyes|face|smile|expression|pose|style|color',
        generated.lower()
    ))

    return {
        "word_count": len(generated.split()),
        "ref_overlap": overlap * 100,
        "colors": colors,
        "visual_details": details,
        "unique_words": len(gen_content),
    }


base_mm_metrics = [compute_caption_metrics(g, r) for g, r in zip(base_mm_responses, real_captions)]
trained_mm_metrics = [compute_caption_metrics(g, r) for g, r in zip(trained_mm_responses, real_captions)]

mm_metric_names = ["word_count", "ref_overlap", "colors", "visual_details", "unique_words"]
mm_labels = ["Word Count", "Ref Overlap %", "Colors Mentioned", "Visual Details", "Unique Words"]

print("%-20s %10s %10s %10s" % ("Metric", "Base", "Trained", "Winner"))
print("-" * 54)
mm_base_avgs = {}
mm_trained_avgs = {}
for name, label in zip(mm_metric_names, mm_labels):
    b = np.mean([m[name] for m in base_mm_metrics])
    t = np.mean([m[name] for m in trained_mm_metrics])
    mm_base_avgs[name] = b
    mm_trained_avgs[name] = t
    w = "Trained" if t > b else ("Base" if b > t else "Tie")
    print("%-20s %10.1f %10.1f %10s" % (label, b, t, w))

eval_results["multimodal"]["base_metrics"] = mm_base_avgs
eval_results["multimodal"]["trained_metrics"] = mm_trained_avgs

### Multimodal Model-as-Judge

We use the **trained LLaVA itself** to judge descriptions.
Given the reference caption, which description is better?

In [ ]:
MM_JUDGE_TEMPLATE = """You are a judge comparing two image descriptions. The image actually shows: {reference}

Description A: {desc_a}

Description B: {desc_b}

Which description is more accurate and detailed? Answer with ONLY the letter A or B."""


def judge_mm_with_model(model, reference, resp_base, resp_trained):
    swap = random.random() < 0.5
    if swap:
        desc_a, desc_b = resp_trained, resp_base
    else:
        desc_a, desc_b = resp_base, resp_trained

    judge_text = MM_JUDGE_TEMPLATE.format(
        reference=reference,
        desc_a=desc_a[:300],
        desc_b=desc_b[:300],
    )

    # Use text-only judging (no image needed for comparing descriptions)
    inputs = processor.tokenizer(
        "USER: " + judge_text + "\nASSISTANT:",
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5,
            temperature=0.1, do_sample=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    answer = processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()

    if "A" in answer and "B" not in answer:
        raw = "A"
    elif "B" in answer and "A" not in answer:
        raw = "B"
    else:
        return "Tie", answer

    if swap:
        winner = "Trained" if raw == "A" else "Base"
    else:
        winner = "Base" if raw == "A" else "Trained"

    return winner, answer


# Run multimodal judge
random.seed(123)
mm_judge_results = []
mm_wins = {"Base": 0, "Trained": 0, "Tie": 0}

print("Model judging %d image description pairs...\n" % len(test_images))
for i in range(len(test_images)):
    winner, raw = judge_mm_with_model(
        model_qlora_mm, real_captions[i],
        base_mm_responses[i], trained_mm_responses[i]
    )
    mm_judge_results.append({"winner": winner, "raw": raw})
    mm_wins[winner] = mm_wins.get(winner, 0) + 1
    print("[%d/%d] %s" % (i + 1, len(test_images), real_captions[i][:60]))
    print("  Winner: %s (raw: %s)" % (winner, raw))

mm_total = len(mm_judge_results)
print("\n" + "=" * 50)
print("MULTIMODAL MODEL-AS-JUDGE RESULTS")
print("=" * 50)
print("Base wins:    %d/%d (%.0f%%)" % (mm_wins["Base"], mm_total, 100 * mm_wins["Base"] / mm_total))
print("Trained wins: %d/%d (%.0f%%)" % (mm_wins["Trained"], mm_total, 100 * mm_wins["Trained"] / mm_total))
print("Ties:         %d/%d (%.0f%%)" % (mm_wins["Tie"], mm_total, 100 * mm_wins["Tie"] / mm_total))

eval_results["multimodal"]["judge_wins"] = mm_wins

In [ ]:
# Multimodal perplexity comparison
mm_base_ppls = []
mm_trained_ppls = []

print("Computing perplexity on image descriptions...\n")
for i in range(len(test_images)):
    base_text = "USER: <image>\nDescribe this image in detail.\nASSISTANT: " + base_mm_responses[i] + "</s>"
    trained_text = "USER: <image>\nDescribe this image in detail.\nASSISTANT: " + trained_mm_responses[i] + "</s>"

    # Text-only perplexity (without image, measures text quality)
    for text, ppl_list, label in [(base_text, mm_base_ppls, "Base"), (trained_text, mm_trained_ppls, "Trained")]:
        inputs = processor.tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model_qlora_mm.device)
        with torch.no_grad():
            outputs = model_qlora_mm(**inputs, labels=inputs["input_ids"])
        ppl_list.append(torch.exp(outputs.loss).item())

    print("[%d] Base PPL: %.1f | Trained PPL: %.1f" % (i + 1, mm_base_ppls[-1], mm_trained_ppls[-1]))

print("\nAvg PPL — Base: %.1f | Trained: %.1f" % (np.mean(mm_base_ppls), np.mean(mm_trained_ppls)))

eval_results["multimodal"]["base_ppl"] = float(np.mean(mm_base_ppls))
eval_results["multimodal"]["trained_ppl"] = float(np.mean(mm_trained_ppls))

In [ ]:
# Multimodal visualization
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Win rate
mm_wl = []
mm_wv = []
mm_wc = []
cmap = {"Base": "#f87171", "Trained": "#34d399", "Tie": "#818cf8"}
for k in ["Base", "Trained", "Tie"]:
    if mm_wins.get(k, 0) > 0:
        mm_wl.append("%s (%d)" % (k, mm_wins[k]))
        mm_wv.append(mm_wins[k])
        mm_wc.append(cmap[k])

axes[0].pie(mm_wv, labels=mm_wl, colors=mm_wc,
            autopct="%.0f%%", textprops={"fontsize": 13, "color": "white"},
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1})
axes[0].set_title("Multimodal Judge Win Rate", fontsize=14)

# 2. Perplexity
x = np.arange(len(test_images))
axes[1].bar(x - 0.2, mm_base_ppls, 0.4, color="#f87171", label="Base", alpha=0.8)
axes[1].bar(x + 0.2, mm_trained_ppls, 0.4, color="#34d399", label="Trained", alpha=0.8)
axes[1].set_title("Perplexity (lower = better)", fontsize=14)
axes[1].set_xlabel("Test Image #")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.2, axis="y")

# 3. Caption metrics
mm_bar_labels = ["Words", "Ref Overlap%", "Colors", "Details", "Unique"]
mm_bar_keys = ["word_count", "ref_overlap", "colors", "visual_details", "unique_words"]
bv = [mm_base_avgs[k] for k in mm_bar_keys]
tv = [mm_trained_avgs[k] for k in mm_bar_keys]
x2 = np.arange(len(mm_bar_labels))
axes[2].bar(x2 - 0.2, bv, 0.4, color="#f87171", label="Base", alpha=0.8)
axes[2].bar(x2 + 0.2, tv, 0.4, color="#34d399", label="Trained", alpha=0.8)
axes[2].set_xticks(x2)
axes[2].set_xticklabels(mm_bar_labels, fontsize=10)
axes[2].set_title("Caption Metrics", fontsize=14)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.show()

### Multimodal Side-by-Side

In [ ]:
import matplotlib.pyplot as plt

n = min(5, len(test_images))
fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n))
if n == 1:
    axes = [axes]

for i in range(n):
    axes[i].imshow(test_images[i]["image"])
    axes[i].axis("off")
    winner = mm_judge_results[i]["winner"] if i < len(mm_judge_results) else "?"
    title = (
        "Real: " + real_captions[i][:80] + "\n"
        "Base: " + base_mm_responses[i][:80] + "\n"
        "Trained: " + trained_mm_responses[i][:80] + "\n"
        "Judge: " + winner
    )
    axes[i].set_title(title, fontsize=10, loc="left", color="white")

plt.suptitle("Base vs QLoRA-Trained Descriptions", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
del model_qlora_mm
clear_gpu()

---
# FINAL DASHBOARD

In [ ]:
from IPython.display import HTML, display

text_wins = eval_results["text"].get("judge_wins", {})
text_total = max(sum(text_wins.values()), 1)
text_dpo_rate = 100 * text_wins.get("DPO", 0) / text_total

mm_wins_f = eval_results["multimodal"].get("judge_wins", {})
mm_total_f = max(sum(mm_wins_f.values()), 1)
mm_trained_rate = 100 * mm_wins_f.get("Trained", 0) / mm_total_f

dashboard = (
    '<div style="font-family:sans-serif;background:#0d1117;padding:20px;border-radius:12px;">'
    '<h1 style="color:#a78bfa;text-align:center;">LLM-as-Judge Evaluation Dashboard</h1>'
    '<p style="color:#888;text-align:center;">All evaluation done locally — no external APIs</p>'
    '<div style="display:flex;gap:20px;margin:20px 0;">'

    # Text card
    '<div style="flex:1;background:#1a1a2e;padding:20px;border-radius:10px;border:1px solid #333;">'
    '<h2 style="color:#818cf8;">Text: TinyLlama 1.1B</h2>'
    '<h3 style="color:#ccc;">Base vs DPO-Aligned</h3>'
    '<div style="font-size:48px;color:#34d399;text-align:center;margin:15px 0;">'
    + ("%.0f%%" % text_dpo_rate) + '</div>'
    '<div style="text-align:center;color:#888;">DPO Win Rate (model judge)</div>'
    '<div style="margin-top:15px;color:#ccc;font-size:13px;">'
    '<div>DPO wins: ' + str(text_wins.get("DPO", 0)) + '</div>'
    '<div>Base wins: ' + str(text_wins.get("Base", 0)) + '</div>'
    '<div>Ties: ' + str(text_wins.get("Tie", 0)) + '</div>'
    '<div style="margin-top:8px;">Avg PPL Base: ' + ("%.0f" % eval_results["text"].get("base_ppl", 0)) + '</div>'
    '<div>Avg PPL DPO: ' + ("%.0f" % eval_results["text"].get("dpo_ppl", 0)) + '</div>'
    '</div></div>'

    # Multimodal card
    '<div style="flex:1;background:#1a1a2e;padding:20px;border-radius:10px;border:1px solid #333;">'
    '<h2 style="color:#818cf8;">Multimodal: LLaVA 7B</h2>'
    '<h3 style="color:#ccc;">Base vs QLoRA-Trained</h3>'
    '<div style="font-size:48px;color:#34d399;text-align:center;margin:15px 0;">'
    + ("%.0f%%" % mm_trained_rate) + '</div>'
    '<div style="text-align:center;color:#888;">Trained Win Rate (model judge)</div>'
    '<div style="margin-top:15px;color:#ccc;font-size:13px;">'
    '<div>Trained wins: ' + str(mm_wins_f.get("Trained", 0)) + '</div>'
    '<div>Base wins: ' + str(mm_wins_f.get("Base", 0)) + '</div>'
    '<div>Ties: ' + str(mm_wins_f.get("Tie", 0)) + '</div>'
    '<div style="margin-top:8px;">Avg PPL Base: ' + ("%.0f" % eval_results["multimodal"].get("base_ppl", 0)) + '</div>'
    '<div>Avg PPL Trained: ' + ("%.0f" % eval_results["multimodal"].get("trained_ppl", 0)) + '</div>'
    '</div></div>'

    '</div>'

    # Evaluation methods table
    '<div style="background:#1a1a2e;padding:20px;border-radius:10px;border:1px solid #333;margin-top:15px;">'
    '<h2 style="color:#a78bfa;">Evaluation Methods Used</h2>'
    '<table style="width:100%;border-collapse:collapse;color:#ccc;">'
    '<tr style="border-bottom:2px solid #4f46e5;">'
    '<th style="text-align:left;padding:10px;">Method</th>'
    '<th style="text-align:left;padding:10px;">Needs API?</th>'
    '<th style="text-align:left;padding:10px;">Speed</th>'
    '<th style="text-align:left;padding:10px;">Best For</th>'
    '</tr>'
    '<tr><td style="padding:8px;color:#f87171;">Automated Metrics</td>'
    '<td style="padding:8px;">No</td><td style="padding:8px;">Instant</td>'
    '<td style="padding:8px;">Quick sanity checks (word count, diversity)</td></tr>'
    '<tr><td style="padding:8px;color:#818cf8;">Rule-Based Scoring</td>'
    '<td style="padding:8px;">No</td><td style="padding:8px;">Instant</td>'
    '<td style="padding:8px;">Consistent rubric (completeness, specificity)</td></tr>'
    '<tr><td style="padding:8px;color:#34d399;">Model-as-Judge</td>'
    '<td style="padding:8px;">No</td><td style="padding:8px;">~1s/pair</td>'
    '<td style="padding:8px;">A/B preference comparison</td></tr>'
    '<tr><td style="padding:8px;color:#f59e0b;">Perplexity</td>'
    '<td style="padding:8px;">No</td><td style="padding:8px;">~0.5s/pair</td>'
    '<td style="padding:8px;">Model confidence / text naturalness</td></tr>'
    '</table></div>'

    '</div>'
)

display(HTML(dashboard))

---
## Key Takeaways

### What We Did
Evaluated both text and multimodal models using **4 methods, zero external APIs**:

| Method | What it measures | Limitation |
|--------|-----------------|------------|
| **Automated Metrics** | Surface features (length, vocab) | Doesn't measure meaning |
| **Rule-Based Scoring** | Rubric dimensions (detail, structure) | Can't judge content accuracy |
| **Model-as-Judge** | Overall preference (A vs B) | Small model = weak judge, self-bias |
| **Perplexity** | Model confidence in text | Trained model always prefers its own outputs |

### Self-Judging Bias
Using a model to judge its own outputs is inherently biased:
- The DPO model will prefer DPO-style responses
- Perplexity will be lower for trained outputs (model was optimized for them)
- This is expected and demonstrates WHY you want an independent judge

### Production Evaluation Stack
```
Automated metrics      <-- cheapest, least reliable
      |
Rule-based scoring     <-- consistent, scalable
      |
Model-as-Judge (self)  <-- what we did here (biased but demonstrates technique)
      |
LLM-as-Judge (strong)  <-- GPT-4 / Claude / Gemini (best automated approach)
      |
Human evaluation       <-- gold standard, expensive
      |
A/B testing (users)    <-- ultimate test
```

### Best Practices
- **Randomize A/B order** — prevents position bias (judges prefer first response)
- **Use multiple methods** — no single metric captures everything
- **Use stronger judges** — swap TinyLlama judge for GPT-4/Claude for real results
- **Compare perplexity trends** — not absolute values (model-dependent)